In [0]:
%sql
-- Create a new table with liquid clustering
-- Unlike partitioning, clustering columns can be changed anytime
CREATE OR REPLACE TABLE fraud_analytics.silver.transactions_clustered
CLUSTER BY (transaction_id, location)
AS SELECT * FROM fraud_analytics.silver.credit_card_fraud;

-- Verify clustering
DESCRIBE DETAIL fraud_analytics.silver.transactions_clustered;

In [0]:
%sql
-- COPY INTO: SQL-based idempotent ingestion
-- Tracks processed files — safe to rerun, never duplicates
CREATE OR REPLACE TABLE fraud_analytics.bronze.copy_into_test
(TransactionID INT, TransactionDate TIMESTAMP, Amount DOUBLE,
 MerchantID INT, TransactionType STRING, Location STRING, IsFraud INT);

COPY INTO fraud_analytics.bronze.copy_into_test
FROM 'abfss://bronze@shruthistorageaccount.dfs.core.windows.net/credit_card_fraud.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

-- Verify
SELECT COUNT(*) FROM fraud_analytics.bronze.copy_into_test;

In [0]:
%sql
-- PIVOT: rows to columns
-- Shows fraud count per transaction type per location
SELECT * FROM (
  SELECT location, transaction_type, 
         CASE WHEN is_fraud = true THEN 1 ELSE 0 END as is_fraud_int
  FROM fraud_analytics.silver.credit_card_fraud
  WHERE location IN ('New York', 'Dallas', 'Chicago')
)
PIVOT (
  SUM(is_fraud_int) 
  FOR transaction_type IN ('purchase', 'refund')
);

In [0]:
# UNPIVOT: columns back to rows using PySpark
from pyspark.sql.functions import expr

df_pivoted = spark.sql("""
    SELECT location, purchase, refund FROM (
        SELECT location, transaction_type,
               CASE WHEN is_fraud = true THEN 1 ELSE 0 END as fraud
        FROM fraud_analytics.silver.credit_card_fraud
        WHERE location IN ('New York', 'Dallas', 'Chicago')
    )
    PIVOT (SUM(fraud) FOR transaction_type IN ('purchase', 'refund'))
""")

# Unpivot back
df_unpivoted = df_pivoted.unpivot(
    ["location"], ["purchase", "refund"], 
    "transaction_type", "fraud_count"
)
df_unpivoted.show()

In [0]:
%sql
-- UNION: combine two datasets (removes duplicates)
SELECT location, COUNT(*) as txn_count 
FROM fraud_analytics.silver.credit_card_fraud
WHERE transaction_year = 2023
GROUP BY location

UNION

SELECT location, COUNT(*) as txn_count
FROM fraud_analytics.silver.credit_card_fraud  
WHERE transaction_year = 2024
GROUP BY location;

-- INTERSECT: rows in BOTH datasets
-- Cities that had fraud in BOTH 2023 and 2024
SELECT DISTINCT location FROM fraud_analytics.silver.credit_card_fraud
WHERE transaction_year = 2023 AND is_fraud = true

INTERSECT

SELECT DISTINCT location FROM fraud_analytics.silver.credit_card_fraud
WHERE transaction_year = 2024 AND is_fraud = true;

-- EXCEPT: rows in first but NOT second
-- Cities with fraud in 2023 but NOT in 2024
SELECT DISTINCT location FROM fraud_analytics.silver.credit_card_fraud
WHERE transaction_year = 2023 AND is_fraud = true

EXCEPT

SELECT DISTINCT location FROM fraud_analytics.silver.credit_card_fraud
WHERE transaction_year = 2024 AND is_fraud = true;

In [0]:
%sql
-- Regular VIEW: virtual table, query runs every time
CREATE OR REPLACE VIEW fraud_analytics.gold.fraud_summary_view AS
SELECT 
    location,
    COUNT(*) as total_transactions,
    SUM(CASE WHEN is_fraud = true THEN 1 ELSE 0 END) as fraud_count,
    ROUND(SUM(CASE WHEN is_fraud = true THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as fraud_rate
FROM fraud_analytics.silver.credit_card_fraud
GROUP BY location;

-- Query the view
SELECT * FROM fraud_analytics.gold.fraud_summary_view 
ORDER BY fraud_rate DESC LIMIT 5;

In [0]:
# Access volume path
dbutils.fs.ls("/Volumes/fraud_analytics/bronze/raw_files/")

In [0]:
# Method 1: Install in notebook (session only)
%pip install faker

# Use it immediately
from faker import Faker
fake = Faker()

# Generate fake transaction data for testing
import pandas as pd
test_data = [{
    "transaction_id": i,
    "merchant_name": fake.company(),
    "city": fake.city(),
    "amount": round(fake.pyfloat(min_value=10, max_value=5000), 2)
} for i in range(5)]

print(pd.DataFrame(test_data).to_string())

In [0]:
%sql
-- Set data retention on Silver table
-- Keep Delta history for 30 days (default is 7)
ALTER TABLE fraud_analytics.silver.credit_card_fraud
SET TBLPROPERTIES (
    'delta.logRetentionDuration' = '30 days',
    'delta.deletedFileRetentionDuration' = '30 days'
);

-- Verify
SHOW TBLPROPERTIES fraud_analytics.silver.credit_card_fraud;

In [0]:
# Updated by Shruthi S — March 25, 2026
# Advanced features: Liquid Clustering, COPY INTO, Pivot/Unpivot

In [0]:
##usage of secret scope through Azure Key vault

dbutils.secrets.get(scope="kv-scope", key="storage-key")